# Confidence Decision Layer

This notebook converts calibrated ResNet50 FT-V2 predictions into product-style decisions: **auto-accept**, **show top-k suggestions**, **request user confirmation**, or **review**. It uses the calibrated prediction artifacts produced by Notebook 4.


## 1. Decision Question

The model now has stable accuracy and improved calibration. The next question is:

> How should a food-recognition product act on each prediction?

This notebook evaluates decision bands using calibrated confidence, top-1/top-2 margin, and known hard-class behavior.


In [ ]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd


In [ ]:
@dataclass(frozen=True)
class CFG:
    """Configuration for confidence decision-layer analysis."""

    INPUT_DIR: Path = Path("/kaggle/working/results/resnet50_error_calibration")
    ARTIFACT_INPUT_DIR: Path = Path(
        "/kaggle/input/food101-resnet50-error-calibration"
    )
    RESULTS_DIR: Path = Path("/kaggle/working/results/confidence_decision_layer")
    PREDICTION_FILE: str = "test_predictions_calibrated.csv"
    HARD_CLASS_FILE: str = "hard_classes_calibrated.csv"
    CONFUSION_FILE: str = "top_confusion_pairs_calibrated.csv"
    MIN_AUTO_ACCEPT_ACCURACY: float = 0.90
    MIN_SUGGEST_ACCURACY: float = 0.80
    AUTO_CONFIDENCE_GRID: tuple[float, ...] = tuple(np.round(np.arange(0.70, 0.96, 0.05), 2))
    SUGGEST_CONFIDENCE_GRID: tuple[float, ...] = tuple(np.round(np.arange(0.35, 0.76, 0.05), 2))
    MARGIN_GRID: tuple[float, ...] = tuple(np.round(np.arange(0.05, 0.51, 0.05), 2))


CFG.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Input directory: {CFG.INPUT_DIR}")
print(f"Artifact fallback directory: {CFG.ARTIFACT_INPUT_DIR}")
print(f"Output directory: {CFG.RESULTS_DIR}")


## 2. Load Calibrated Outputs

The notebook first looks for Notebook 4 outputs in the current Kaggle working directory. If the outputs have been uploaded as a Kaggle Dataset, set `CFG.ARTIFACT_INPUT_DIR` to that mounted path.


In [ ]:
def resolve_input_file(filename: str) -> Path:
    """Resolve an input artifact from working output or an attached dataset."""
    candidates = [
        CFG.INPUT_DIR / filename,
        CFG.ARTIFACT_INPUT_DIR / filename,
    ]
    for root in [CFG.INPUT_DIR, CFG.ARTIFACT_INPUT_DIR]:
        if root.exists():
            matches = sorted(root.rglob(filename))
            candidates.extend(matches)

    for candidate in candidates:
        if candidate.exists():
            return candidate

    searched = "\n".join(str(candidate) for candidate in candidates[:2])
    raise FileNotFoundError(f"Could not find {filename}. Searched:\n{searched}")


prediction_path = resolve_input_file(CFG.PREDICTION_FILE)
hard_class_path = resolve_input_file(CFG.HARD_CLASS_FILE)
confusion_path = resolve_input_file(CFG.CONFUSION_FILE)

predictions_df = pd.read_csv(prediction_path)
hard_classes_df = pd.read_csv(hard_class_path)
confusion_pairs_df = pd.read_csv(confusion_path)

print(f"Predictions: {prediction_path}")
print(f"Hard classes: {hard_class_path}")
print(f"Confusion pairs: {confusion_path}")
print(f"Rows: {len(predictions_df):,}")
predictions_df.head()


## 3. Feature Engineering

Decision bands use calibrated top-1 confidence, top-1/top-2 confidence margin, hard-class membership, and repeated confusion-pair membership.


In [ ]:
def parse_confidence_list(value: str) -> list[float]:
    """Parse pipe-separated confidence values from Notebook 4."""
    return [float(item) for item in str(value).split("|") if item != ""]


def parse_label_list(value: str) -> list[str]:
    """Parse pipe-separated class labels from Notebook 4."""
    return [item for item in str(value).split("|") if item != ""]


hard_class_names = set(hard_classes_df["class_name"].tolist())
frequent_confusion_pairs = set(
    zip(confusion_pairs_df["actual"], confusion_pairs_df["predicted"])
)

features_df = predictions_df.copy()
features_df["top_5_labels"] = features_df["top_5"].apply(parse_label_list)
features_df["top_5_confidences"] = features_df["top_5_confidence"].apply(parse_confidence_list)
features_df["top_1_confidence"] = features_df["top_5_confidences"].apply(lambda values: values[0])
features_df["top_2_confidence"] = features_df["top_5_confidences"].apply(
    lambda values: values[1] if len(values) > 1 else 0.0
)
features_df["top_1_top_2_margin"] = (
    features_df["top_1_confidence"] - features_df["top_2_confidence"]
)
features_df["is_hard_actual_class"] = features_df["actual"].isin(hard_class_names)
features_df["is_hard_predicted_class"] = features_df["predicted"].isin(hard_class_names)
features_df["is_hard_case"] = (
    features_df["is_hard_actual_class"] | features_df["is_hard_predicted_class"]
)
features_df["is_frequent_confusion_pair"] = features_df.apply(
    lambda row: (row["actual"], row["predicted"]) in frequent_confusion_pairs,
    axis=1,
)
features_df["top_5_contains_actual"] = features_df.apply(
    lambda row: row["actual"] in row["top_5_labels"],
    axis=1,
)

features_df.to_csv(CFG.RESULTS_DIR / "decision_features.csv", index=False)
features_df[[
    "actual",
    "predicted",
    "top_1_confidence",
    "top_1_top_2_margin",
    "is_correct",
    "top_5_contains_actual",
    "is_hard_case",
    "is_frequent_confusion_pair",
]].head()


## 4. Threshold Search

Search simple thresholds that create interpretable decision bands. The goal is not to hide errors; it is to understand which predictions are safe to accept automatically and which should become suggestions or confirmation prompts.


In [ ]:
def assign_decision_band(
    row: pd.Series,
    auto_confidence: float,
    suggest_confidence: float,
    margin_threshold: float,
) -> str:
    """Assign a product decision band to one prediction."""
    if row["is_frequent_confusion_pair"]:
        return "review"
    if row["is_hard_case"] and row["top_1_confidence"] < auto_confidence:
        return "confirm"
    if (
        row["top_1_confidence"] >= auto_confidence
        and row["top_1_top_2_margin"] >= margin_threshold
        and not row["is_hard_case"]
    ):
        return "auto_accept"
    if row["top_1_confidence"] >= suggest_confidence and row["top_5_contains_actual"]:
        return "suggest"
    return "confirm"


def decision_band_metrics(decision_df: pd.DataFrame) -> pd.DataFrame:
    """Summarize coverage and accuracy by decision band."""
    rows = []
    total = len(decision_df)
    for band, band_df in decision_df.groupby("decision_band"):
        rows.append(
            {
                "decision_band": band,
                "sample_count": len(band_df),
                "coverage": len(band_df) / total,
                "top_1_accuracy": band_df["is_correct"].mean(),
                "top_5_contains_actual": band_df["top_5_contains_actual"].mean(),
                "mean_confidence": band_df["top_1_confidence"].mean(),
                "mean_margin": band_df["top_1_top_2_margin"].mean(),
            }
        )
    return pd.DataFrame(rows).sort_values("decision_band")


def evaluate_policy(auto_confidence: float, suggest_confidence: float, margin_threshold: float) -> dict:
    """Evaluate one threshold policy."""
    decision_df = features_df.copy()
    decision_df["decision_band"] = decision_df.apply(
        assign_decision_band,
        axis=1,
        auto_confidence=auto_confidence,
        suggest_confidence=suggest_confidence,
        margin_threshold=margin_threshold,
    )
    metrics_df = decision_band_metrics(decision_df)
    metrics = {
        "auto_confidence": auto_confidence,
        "suggest_confidence": suggest_confidence,
        "margin_threshold": margin_threshold,
    }
    for _, row in metrics_df.iterrows():
        band = row["decision_band"]
        metrics[f"{band}_coverage"] = row["coverage"]
        metrics[f"{band}_top_1_accuracy"] = row["top_1_accuracy"]
        metrics[f"{band}_top_5_contains_actual"] = row["top_5_contains_actual"]
    return metrics


policy_rows = []
for auto_confidence in CFG.AUTO_CONFIDENCE_GRID:
    for suggest_confidence in CFG.SUGGEST_CONFIDENCE_GRID:
        if suggest_confidence >= auto_confidence:
            continue
        for margin_threshold in CFG.MARGIN_GRID:
            policy_rows.append(
                evaluate_policy(auto_confidence, suggest_confidence, margin_threshold)
            )

policy_search_df = pd.DataFrame(policy_rows).fillna(0.0)
policy_search_df["meets_auto_accuracy"] = (
    policy_search_df["auto_accept_top_1_accuracy"] >= CFG.MIN_AUTO_ACCEPT_ACCURACY
)
policy_search_df["meets_suggest_accuracy"] = (
    policy_search_df["suggest_top_5_contains_actual"] >= CFG.MIN_SUGGEST_ACCURACY
)
policy_search_df["policy_score"] = (
    2.0 * policy_search_df["auto_accept_coverage"]
    + policy_search_df["suggest_coverage"]
    - 0.5 * policy_search_df["review_coverage"]
)
eligible_policy_df = policy_search_df[
    policy_search_df["meets_auto_accuracy"]
    & policy_search_df["meets_suggest_accuracy"]
].copy()

if eligible_policy_df.empty:
    best_policy = policy_search_df.sort_values("policy_score", ascending=False).iloc[0]
    print("No policy met all minimum targets; using highest score policy.")
else:
    best_policy = eligible_policy_df.sort_values("policy_score", ascending=False).iloc[0]

policy_search_df.to_csv(CFG.RESULTS_DIR / "decision_policy_search.csv", index=False)
best_policy.to_frame().T


## 5. Final Decision Policy

Apply the selected thresholds and export band-level metrics plus examples. These files can be used directly in a product discussion or demo.


In [ ]:
AUTO_CONFIDENCE = float(best_policy["auto_confidence"])
SUGGEST_CONFIDENCE = float(best_policy["suggest_confidence"])
MARGIN_THRESHOLD = float(best_policy["margin_threshold"])

decision_df = features_df.copy()
decision_df["decision_band"] = decision_df.apply(
    assign_decision_band,
    axis=1,
    auto_confidence=AUTO_CONFIDENCE,
    suggest_confidence=SUGGEST_CONFIDENCE,
    margin_threshold=MARGIN_THRESHOLD,
)
band_metrics_df = decision_band_metrics(decision_df)
policy_df = pd.DataFrame(
    [
        {
            "auto_confidence": AUTO_CONFIDENCE,
            "suggest_confidence": SUGGEST_CONFIDENCE,
            "margin_threshold": MARGIN_THRESHOLD,
            "min_auto_accept_accuracy": CFG.MIN_AUTO_ACCEPT_ACCURACY,
            "min_suggest_accuracy": CFG.MIN_SUGGEST_ACCURACY,
        }
    ]
)

decision_df.to_csv(CFG.RESULTS_DIR / "test_predictions_with_decisions.csv", index=False)
band_metrics_df.to_csv(CFG.RESULTS_DIR / "decision_band_metrics.csv", index=False)
policy_df.to_csv(CFG.RESULTS_DIR / "decision_policy.csv", index=False)

for band in ["auto_accept", "suggest", "confirm", "review"]:
    examples = decision_df[decision_df["decision_band"] == band].head(25)
    examples.to_csv(CFG.RESULTS_DIR / f"decision_examples_{band}.csv", index=False)

print("Decision policy")
display(policy_df)
print("Decision band metrics")
display(band_metrics_df)


## 6. Decision Wrapper

Use this helper after inference to attach the product action to any top-k prediction result that follows the Notebook 4 schema.


In [ ]:
def recommend_action(prediction_row: pd.Series) -> str:
    """Return the decision action for one prediction row."""
    feature_row = prediction_row.copy()
    if "top_5_confidences" not in feature_row:
        feature_row["top_5_confidences"] = parse_confidence_list(feature_row["top_5_confidence"])
    if "top_5_labels" not in feature_row:
        feature_row["top_5_labels"] = parse_label_list(feature_row["top_5"])
    feature_row["top_1_confidence"] = feature_row["top_5_confidences"][0]
    feature_row["top_2_confidence"] = (
        feature_row["top_5_confidences"][1]
        if len(feature_row["top_5_confidences"]) > 1
        else 0.0
    )
    feature_row["top_1_top_2_margin"] = (
        feature_row["top_1_confidence"] - feature_row["top_2_confidence"]
    )
    feature_row["is_hard_case"] = (
        feature_row["actual"] in hard_class_names
        or feature_row["predicted"] in hard_class_names
    )
    feature_row["is_frequent_confusion_pair"] = (
        feature_row["actual"], feature_row["predicted"]
    ) in frequent_confusion_pairs
    feature_row["top_5_contains_actual"] = feature_row["actual"] in feature_row["top_5_labels"]

    return assign_decision_band(
        feature_row,
        auto_confidence=AUTO_CONFIDENCE,
        suggest_confidence=SUGGEST_CONFIDENCE,
        margin_threshold=MARGIN_THRESHOLD,
    )


sample = predictions_df.iloc[0].copy()
print(f"Recommended action: {recommend_action(sample)}")


## 7. Run Insight

This notebook translates model metrics into product behavior. The key output is not a new model checkpoint; it is a policy for deciding when to trust the model, when to show suggestions, and when to ask the user for confirmation.
